In [1]:
import requests
import json
from langsmith import traceable
from embed_utils import embed_text_openai
import os
WEAVIATE_URL = "https://1exconarfknip3xtkwcg.c0.us-east1.gcp.weaviate.cloud"  # Your Weaviate instance URL
WEAVIATE_API_KEY = os.environ.get("WEAVIATE_API_KEY")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

SCHEMA_API_URL = f"{WEAVIATE_URL}/v1/schema"
BATCH_API_URL = f"{WEAVIATE_URL}/v1/batch/objects"
BATCH_SIZE = 100

COLLECTION_NAME = "SWD_passages_openai"

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {WEAVIATE_API_KEY}"
}

@traceable
def get_weaviate_vdb_results(embedded_query, weaviate_url, filter_obj=None):
    query = """
    query($vector: [Float!]!) {
      Get {
        SWD_passages_openai (
          limit: 10
          nearVector: {
            vector: $vector
          }
          where: $filter_obj 
        ) {
          english_text
          hebrew_text
          book_name
          page_number
          passage_id
          translation_id
          text_to_embed
        }
      }
    }
    """

    variables = {
        "vector": embedded_query,
        "filter_obj": filter_obj if filter_obj else ""
    }

    response = requests.post(
        f"{weaviate_url}/v1/graphql",
        headers=headers,
        json={"query": query, "variables": variables}
    )
    print(json.dumps(response.json(), indent=2))
    response.raise_for_status()
    results = response.json()

    passages = [
        {
            'passage_id': result['passage_id'],
            'hebrew_text': result['hebrew_text'],
            'english_text': result['english_text'],
            'translation_id': result['translation_id'],
            'book_name': result['book_name'],
            'page_number': result['page_number'],
            'text_to_embed': result['text_to_embed']
        }
        for result in results['data']['Get']['Question']
    ]

    # Filter out passages that have English text which includes "sample translation"
    passages = [passage for passage in passages if "sample translation" not in passage['english_text'].lower()]
    return passages

/Users/eliplutchok/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [53]:
# Sample query
query_text = "charity"
# Embed the query
embedded_query = embed_text_openai(query_text)
print(len(embedded_query))
# Call the function
results = get_weaviate_vdb_results(
    embedded_query=embedded_query,
    weaviate_url=WEAVIATE_URL,
   
)

# Print the results
print(f"Number of results: {len(results)}")
for i, result in enumerate(results, 1):
    print(f"\nResult {i}:")
    print(f"Book: {result['book_name']}")
    print(f"Page: {result['page_number']}")
    print(f"English Text: {result['english_text'][:100]}...")  # Print first 100 characters


1536
{
  "errors": [
    {
      "locations": [
        {
          "column": 18,
          "line": 9
        },
        {
          "column": 5,
          "line": 2
        }
      ],
      "message": "Variable \"$filter_obj\" is not defined.",
      "path": null
    }
  ]
}


KeyError: 'data'